In [0]:
# Databricks notebook source
# MAGIC %md
# MAGIC # 04 - Monitoring & Maintenance
# MAGIC
# MAGIC This notebook is not part of the critical path - it's what you'd schedule separately
# MAGIC (e.g. daily) or wire into a dashboard/alert to keep the pipeline healthy in production.
# MAGIC It demonstrates the kind of checks discussed in the README under "How would you
# MAGIC monitor this in production".

# COMMAND ----------

CATALOG = "lakehouse_demo"
SCHEMA = "transactions"
spark.sql(f"USE CATALOG {CATALOG}")
spark.sql(f"USE SCHEMA {SCHEMA}")

BRONZE_TABLE = f"{CATALOG}.{SCHEMA}.bronze_transactions"
SILVER_TABLE = f"{CATALOG}.{SCHEMA}.silver_transactions"
QUARANTINE_TABLE = f"{CATALOG}.{SCHEMA}.silver_quarantine"
GOLD_MONTHLY = f"{CATALOG}.{SCHEMA}.gold_revenue_by_month"

# COMMAND ----------

# MAGIC %md
# MAGIC ## 1. Row-count reconciliation across layers
# MAGIC A big drop between Bronze -> Silver row counts (beyond what quarantine explains) is a
# MAGIC red flag worth alerting on.

# COMMAND ----------

from pyspark.sql import functions as F

bronze_n = spark.table(BRONZE_TABLE).count()
silver_n = spark.table(SILVER_TABLE).count()
quarantine_n = spark.table(QUARANTINE_TABLE).count()

print(f"Bronze rows:      {bronze_n}")
print(f"Silver rows:      {silver_n}")
print(f"Quarantined rows: {quarantine_n}")
print(f"Unexplained gap:  {bronze_n - silver_n - quarantine_n}  (should be ~0, small variance ok due to in-batch dedup)")

# COMMAND ----------

# MAGIC %md
# MAGIC ## 2. Data quality trend
# MAGIC Track the quarantine rate over time - a sudden spike usually means an upstream schema
# MAGIC or data-contract change on the client's side.

# COMMAND ----------

display(spark.sql(f"""
    SELECT date(_quarantined_ts) AS day, count(*) AS rejected_rows
    FROM {QUARANTINE_TABLE}
    GROUP BY date(_quarantined_ts)
    ORDER BY day
"""))

# COMMAND ----------

# MAGIC %md
# MAGIC ## 3. Freshness check
# MAGIC How stale is Gold relative to now? Useful as a Databricks SQL alert
# MAGIC ("fail if max(last_updated_ts) < now() - 26 hours" for a daily job, for example).

# COMMAND ----------

display(spark.sql(f"""
    SELECT max(last_updated_ts) AS gold_last_updated,
           round((unix_timestamp(current_timestamp()) - unix_timestamp(max(last_updated_ts))) / 3600, 1) AS hours_since_update
    FROM {GOLD_MONTHLY}
"""))

# COMMAND ----------

# MAGIC %md
# MAGIC ## 4. Streaming job history / throughput
# MAGIC In a real workspace you'd look at:
# MAGIC - **Job runs UI** (Workflows > Job > Runs) for success/failure/duration trends and to set
# MAGIC   up email/Slack/webhook alerts on failure
# MAGIC - `DESCRIBE HISTORY <table>` for a Delta-native audit trail of every write (who/what/when)
# MAGIC - System tables (`system.access.audit`, `system.billing.usage`,
# MAGIC   `system.lakeflow.job_run_timeline` where available) for cross-pipeline cost/reliability
# MAGIC   reporting
# MAGIC - A `StreamingQueryListener` (for long-running streams) or the `.lastProgress` /
# MAGIC   `.recentProgress` of each `writeStream` query, written to a small `pipeline_run_log`
# MAGIC   Delta table, to track rows-processed and batch duration per run over time

# COMMAND ----------

display(spark.sql(f"DESCRIBE HISTORY {SILVER_TABLE} LIMIT 10"))

# COMMAND ----------

# MAGIC %md
# MAGIC ## 5. Table maintenance
# MAGIC `OPTIMIZE` compacts small files (important since Auto Loader + streaming MERGE both tend
# MAGIC to produce many small files); `VACUUM` reclaims space from files no longer referenced by
# MAGIC the current table version, after the retention window. Schedule weekly, not per-run.

# COMMAND ----------

for tbl in [BRONZE_TABLE, SILVER_TABLE, GOLD_MONTHLY]:
    spark.sql(f"OPTIMIZE {tbl}")

# Default retention is 7 days - only shorten this deliberately, and never below the checkpoint
# interval of any downstream stream reading via CDF/time-travel from this table.
# spark.sql(f"VACUUM {SILVER_TABLE}")


In [0]:
path = '/Volumes/lakehouse_demo/transactions/landing'


In [0]:
spark.sql(f'''
          select * 
          from json.`{path}/json`''').display()